# MKM411 — Computational Fluid Dynamics
## Lecture: Scientific Machine Learning for Fluid Dynamics
### Part 1 of 2 — Neural Networks & the SciML Landscape
---
**Department of Mechanical & Aeronautical Engineering | University of Pretoria**  
*Prof K.J. Craig & Prof M. Bhamjee*


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import warnings
warnings.filterwarnings('ignore')

from lecture_utils import (UP_BLUE, UP_GOLD, ACCENT, SEED,
                            SmallNet, DemoANN,
                            draw_network, plot_sciml_landscape,
                            plot_loss_comparison,
                            activation_explorer,
                            forward_pass_widget,
                            loss_landscape_widget)

torch.manual_seed(SEED)
np.random.seed(SEED)
print("Environment ready.")


---
## 1. Motivation — Why is Machine Learning Entering CFD?

Classical CFD methods (FDM, FVM, FEM) are mature, reliable, and physically grounded.
So why are researchers increasingly turning to machine learning?

### The core tension

| Challenge | Classical CFD | Implication |
|-----------|--------------|-------------|
| **High-dimensional parameter spaces** | Re-solve for every new set | Design optimisation requires thousands of solves |
| **Real-time prediction** | Minutes to hours per solve | Not viable for control or digital twins |
| **Noisy or incomplete data** | Needs clean BCs and ICs | Experimental data is messy |
| **Inverse problems** | Hard to formulate | Cannot easily infer unknown fields |

### What ML offers
- **Surrogates** — fast approximations of expensive solvers
- **Data assimilation** — learn from experimental observations
- **Generalisation** — one trained model predicts across a parameter space
- **Inverse problem solving** — infer unknowns from observable data

### What ML does *not* replace
Physics. A purely data-driven model trained on limited data will violate
conservation laws, produce non-physical results, and fail to generalise.
This is the central motivation for **Scientific Machine Learning** — embedding
physical knowledge into the learning process.


In [ ]:
# Curse of dimensionality — how many solver runs does a parameter sweep require?
dims = np.arange(1, 9)
points_per_dim = 10

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(dims, points_per_dim**dims, color=UP_BLUE, alpha=0.8)
ax.set_yscale('log')
ax.set_xlabel('Number of parameters', fontsize=12)
ax.set_ylabel('Solver runs required', fontsize=12)
ax.set_title('Curse of Dimensionality — 10 values per parameter',
             fontsize=12, color=UP_BLUE)
ax.set_xticks(dims)
ax.grid(True, alpha=0.3, axis='y')

for d, label in {1: r'vary $rho$', 3: r'vary $rho, c_p, k$',
                  6: 'vary 6 BCs'}.items():
    ax.text(d, points_per_dim**d * 2, label, ha='center',
            fontsize=8, color='gray')
plt.tight_layout(); plt.show()
print(f"6 parameters × 10 values = {10**6:,} solver runs")
print(f"At 1 min/solve: {10**6/60/24:.0f} days of compute")


---
## 2. Artificial Neural Networks — Architecture

A neural network is a parametric function $f_\theta: \mathbb{R}^n \to \mathbb{R}^m$
composed of layers of simple operations.

### The fully-connected layer

$$z_i^{(l)} = \sum_{j} w_{ij}^{(l)} a_j^{(l-1)} + b_i^{(l)}, \qquad a_i^{(l)} = \sigma(z_i^{(l)})$$

### Universal approximation theorem
A network with a single hidden layer of sufficient width can approximate
any continuous function on a compact domain to arbitrary precision.
In practice, **depth** is more efficient than width alone.

### The project network
Input: $(\hat{x}, \hat{y}, \hat{t}, \hat{\rho}, \hat{c}_p, \hat{k})$ — 6 inputs  
Architecture: 5 hidden layers × 40 neurons — **6,881 parameters**  
Output: $T$ [K]


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
draw_network(ax, [6, 8, 8, 8, 8, 8, 1],
             title='Project Network: 6 → [40x5] → 1   (shown with 8 neurons for clarity)')
ax.text(0.0,  1.02, 'x,y,t\nrho,cp,k', ha='center', fontsize=8, color=UP_BLUE,
        transform=ax.transAxes)
ax.text(1.0,  1.02, 'T [K]', ha='center', fontsize=8, color=UP_GOLD,
        transform=ax.transAxes)
plt.tight_layout(); plt.show()


---
## 3. Activation Functions

Activation functions introduce **non-linearity**. Without them, any stack of
linear layers collapses to a single linear transformation.

### The PINN constraint
PINNs compute $\partial^2 T / \partial x^2$ via autograd. The activation must
be **at least twice differentiable** with **non-zero second derivative** everywhere.

**`tanh` satisfies this. `ReLU` does not** — its second derivative is zero almost everywhere.


In [ ]:
# Interactive activation function explorer
# Toggle functions, show derivatives, highlight the PINN requirement
activation_explorer()


---
## 4. Training — Learning from Data

### Loss function (data-driven ANN)
$$\mathcal{L}_{\text{ANN}}(\theta) = \frac{1}{N} \sum_{i=1}^{N} \left( f_\theta(\mathbf{x}_i) - T_i \right)^2$$

### Gradient descent
$$\theta \leftarrow \theta - \eta \, \nabla_\theta \mathcal{L}$$

### Adam optimiser
Adapts the learning rate per parameter using first and second moment estimates
of the gradient — more robust than fixed learning rate methods.

$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t, \qquad v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$$
$$\theta_t = \theta_{t-1} - \eta \, \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$


In [ ]:
# Interactive forward pass — see how inputs propagate through layers
forward_pass_widget()


In [ ]:
# Interactive loss landscape — gradient descent in action
# Drag the weight slider, take gradient steps, adjust learning rate
loss_landscape_widget()


---
## 5. The SciML Landscape — Where ANNs Fit

**Scientific Machine Learning** embeds physical knowledge into the learning process.

| Approach | Key idea |
|---------|---------|
| **Data-driven surrogate** | Learn a fast emulator from simulation data |
| **PINNs** | Embed PDE in the loss function |
| **Neural Operators** | Learn mappings between function spaces (DeepONet, FNO) |
| **GNNs** | Learn on unstructured meshes |
| **GANs** | Learn the distribution of flow fields |
| **Foundation Models** | Large pre-trained models (Aurora, Poseidon) |

### The inverse problem — a teaser
Standard CFD: given BCs and ICs, find the flow field.  
PINNs: given sparse observations of *one* field, infer *all* fields.  
Raissi et al. (2020) inferred velocity and pressure from concentration measurements alone.
We will see this in Part 2.


In [ ]:
fig = plot_sciml_landscape(highlight_pinn=True)
plt.show()


---
## Summary — Part 1

| Concept | Key takeaway |
|---------|-------------|
| **Why SciML** | Classical CFD is expensive; ML offers fast surrogates and generalisation |
| **ANN architecture** | Layers of weighted sums + non-linear activations |
| **Activation function** | Must be twice differentiable for PINNs — use `tanh` |
| **Training** | Minimise loss via backpropagation + Adam |
| **SciML landscape** | Spectrum from pure physics to pure data — PINNs sit in the middle |

**In Part 2:** We embed the Navier-Stokes equations into the loss function
and apply this to the Von Karman vortex street behind a cylinder.
